In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Libraries loaded! ✅")

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('diabetes.csv')
print("Shape:", df.shape)
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)
print("\nBasic statistics:")
df.describe()

In [ ]:
print("Outcome counts:")
print(df['Outcome'].value_counts())

plt.figure(figsize=(6,4))
df['Outcome'].value_counts().plot(kind='bar', color=['steelblue', 'coral'])
plt.title('Diabetes vs No Diabetes')
plt.xlabel('Outcome (0=No Diabetes, 1=Diabetes)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df.hist(figsize=(12,10), bins=20, color='steelblue', edgecolor='black')
plt.suptitle('Distribution of All Features')
plt.tight_layout()
plt.show()

In [ ]:
# Replace 0s with median in columns where 0 is impossible
cols_to_fix = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in cols_to_fix:
    median_val = df[col].median()
    df[col] = df[col].replace(0, median_val)
    print(f"{col}: replaced 0s with median {median_val}")

print("\nCleaning done! ✅")
df.describe()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Split features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train the model
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)

print("Model trained! ✅")
print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes','Diabetes'],
            yticklabels=['No Diabetes','Diabetes'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
def predict_diabetes(pregnancies, glucose, blood_pressure,
                     skin_thickness, insulin, bmi, dpf, age):
    input_data = [[pregnancies, glucose, blood_pressure,
                   skin_thickness, insulin, bmi, dpf, age]]
    input_scaled = scaler.transform(input_data)
    result = model.predict(input_scaled)[0]
    prob = model.predict_proba(input_scaled)[0]

    print(f"Prediction: {'⚠️ Diabetes Detected' if result == 1 else '✅ No Diabetes'}")
    print(f"Confidence: {round(max(prob) * 100, 2)}%")

# Test with a patient
predict_diabetes(
    pregnancies=6,
    glucose=148,
    blood_pressure=72,
    skin_thickness=35,
    insulin=100,
    bmi=33.6,
    dpf=0.627,
    age=50
)

In [ ]:
def predict_diabetes(pregnancies, glucose, blood_pressure,
                     skin_thickness, insulin, bmi, dpf, age):

    input_data = pd.DataFrame([[pregnancies, glucose, blood_pressure,
                   skin_thickness, insulin, bmi, dpf, age]],
                   columns=X.columns)
    input_scaled = scaler.transform(input_data)
    result = model.predict(input_scaled)[0]
    prob = model.predict_proba(input_scaled)[0]

    print(f"Prediction: {'⚠️ Diabetes Detected' if result == 1 else '✅ No Diabetes'}")
    print(f"Confidence: {round(max(prob) * 100, 2)}%")

# Test with a healthy patient
predict_diabetes(
    pregnancies=1,
    glucose=85,
    blood_pressure=66,
    skin_thickness=29,
    insulin=30,
    bmi=26.6,
    dpf=0.351,
    age=25
)

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

# Evaluate
rf_pred = rf_model.predict(X_test)
print("Random Forest Accuracy:", round(accuracy_score(y_test, rf_pred) * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

In [ ]:
feature_importance = pd.Series(rf_model.feature_importances_, index=df.drop('Outcome', axis=1).columns)
feature_importance = feature_importance.sort_values(ascending=False)

plt.figure(figsize=(10,6))
feature_importance.plot(kind='bar', color='steelblue')
plt.title('Feature Importance - Which factors predict Diabetes most?')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
models = ['Logistic Regression', 'Random Forest']
accuracies = [68.83, 76.62]

plt.figure(figsize=(8,5))
plt.bar(models, accuracies, color=['steelblue', 'coral'])
plt.title('Model Comparison - Accuracy')
plt.ylabel('Accuracy (%)')
plt.ylim(60, 85)
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.3, f'{v}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import joblib

# Save the Random Forest model
joblib.dump(rf_model, 'diabetes_rf_model.pkl')

# Save the scaler too (very important!)
joblib.dump(scaler, 'diabetes_scaler.pkl')

print("Model saved! ✅")
print("Scaler saved! ✅")

In [ ]:
from google.colab import files

files.download('diabetes_rf_model.pkl')
files.download('diabetes_scaler.pkl')

In [ ]:
# Load the saved model
loaded_model = joblib.load('diabetes_rf_model.pkl')
loaded_scaler = joblib.load('diabetes_scaler.pkl')

# Test it works
input_data = pd.DataFrame([[6, 148, 72, 35, 100, 33.6, 0.627, 50]],
                   columns=X.columns)
input_scaled = loaded_scaler.transform(input_data)
result = loaded_model.predict(input_scaled)[0]

print("Model loaded successfully! ✅")
print(f"Test prediction: {'⚠️ Diabetes Detected' if result == 1 else '✅ No Diabetes'}")

In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
app_code = """
import streamlit as st
import joblib
import pandas as pd

# Load model and scaler
model = joblib.load('diabetes_rf_model.pkl')
scaler = joblib.load('diabetes_scaler.pkl')

st.title('🏥 Diabetes Prediction App')
st.write('Enter patient details below to predict diabetes risk')

# Input fields
pregnancies = st.slider('Pregnancies', 0, 17, 3)
glucose = st.slider('Glucose Level', 44, 199, 117)
blood_pressure = st.slider('Blood Pressure', 24, 122, 72)
skin_thickness = st.slider('Skin Thickness', 7, 99, 23)
insulin = st.slider('Insulin', 14, 846, 30)
bmi = st.slider('BMI', 18.2, 67.1, 32.0)
dpf = st.slider('Diabetes Pedigree Function', 0.078, 2.42, 0.47)
age = st.slider('Age', 21, 81, 33)

if st.button('Predict'):
    input_data = pd.DataFrame([[pregnancies, glucose, blood_pressure,
                                skin_thickness, insulin, bmi, dpf, age]],
                               columns=['Pregnancies','Glucose','BloodPressure',
                                        'SkinThickness','Insulin','BMI',
                                        'DiabetesPedigreeFunction','Age'])
    input_scaled = scaler.transform(input_data)
    result = model.predict(input_scaled)[0]
    prob = model.predict_proba(input_scaled)[0]

    if result == 1:
        st.error(f'⚠️ Diabetes Detected! Confidence: {round(max(prob)*100, 2)}%')
    else:
        st.success(f'✅ No Diabetes! Confidence: {round(max(prob)*100, 2)}%')
"""

with open('app.py', 'w') as f:
    f.write(app_code)

print("App file created! ✅")

In [ ]:
!pip install streamlit -q
!npm install -g localtunnel

import time
import subprocess
import threading

# Run streamlit in background
def run_app():
    subprocess.run(['streamlit', 'run', 'app.py', '--server.headless', 'true'])

t = threading.Thread(target=run_app)
t.daemon = True
t.start()

# Wait for streamlit to start
time.sleep(10)
print("Streamlit is running!")

# Create public URL
!lt --port 8501

In [ ]:
# Kill existing processes
import subprocess
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)

import time
time.sleep(3)

# Restart with better settings
!streamlit run app.py --server.headless true --server.enableCORS false --server.enableXsrfProtection false &
!sleep 8
!lt --port 8501

In [ ]:
import subprocess
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
subprocess.run(['pkill', '-f', 'lt'], capture_output=True)
print("Cleared! ✅")

In [ ]:
!pip install streamlit -q

# Write the app
import subprocess
import threading
import time
import os

# Start streamlit
os.system("streamlit run app.py --server.port 8501 --server.headless true &")
time.sleep(10)

# Use colab's built in method
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8501)"))

In [ ]:
import os
import time

# Kill any existing streamlit
os.system("pkill -f streamlit")
time.sleep(3)

# Start streamlit fresh
os.system("streamlit run app.py --server.port 8501 --server.headless true &")

# Wait longer this time
print("Waiting for Streamlit to start...")
for i in range(20):
    time.sleep(1)
    print(f"{i+1} seconds...")

print("Done! Getting URL...")
from google.colab.output import eval_js
print("\nYour app URL:")
print(eval_js("google.colab.kernel.proxyPort(8501)"))

In [ ]:
!cat /content/logs.txt

In [ ]:
!cat app.py

In [ ]:
app_code = """
import streamlit as st
import joblib
import pandas as pd

# Load model and scaler
model = joblib.load('diabetes_rf_model.pkl')
scaler = joblib.load('diabetes_scaler.pkl')

st.title('🏥 Diabetes Prediction App')
st.write('Enter patient details below to predict diabetes risk')

# Input fields
pregnancies = st.slider('Pregnancies', 0, 17, 3)
glucose = st.slider('Glucose Level', 44, 199, 117)
blood_pressure = st.slider('Blood Pressure', 24, 122, 72)
skin_thickness = st.slider('Skin Thickness', 7, 99, 23)
insulin = st.slider('Insulin', 14, 846, 30)
bmi = st.slider('BMI', 18.2, 67.1, 32.0)
dpf = st.slider('Diabetes Pedigree Function', 0.078, 2.42, 0.47)
age = st.slider('Age', 21, 81, 33)

if st.button('Predict'):
    input_data = pd.DataFrame([[pregnancies, glucose, blood_pressure,
                                skin_thickness, insulin, bmi, dpf, age]],
                               columns=['Pregnancies','Glucose','BloodPressure',
                                        'SkinThickness','Insulin','BMI',
                                        'DiabetesPedigreeFunction','Age'])
    input_scaled = scaler.transform(input_data)
    result = model.predict(input_scaled)[0]
    prob = model.predict_proba(input_scaled)[0]

    if result == 1:
        st.error(f'⚠️ Diabetes Detected! Confidence: {round(max(prob)*100, 2)}%')
    else:
        st.success(f'✅ No Diabetes! Confidence: {round(max(prob)*100, 2)}%')
"""

with open('app.py', 'w') as f:
    f.write(app_code)

print("app.py created! ✅")
!cat app.py

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true 2>&1 | head -50

In [ ]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8501)"))

In [ ]:
import os
import time
import threading

# Kill existing streamlit
os.system("pkill -f streamlit")
time.sleep(2)

# Run streamlit in background
def run():
    os.system("streamlit run app.py --server.port 8501 --server.headless true > /content/logs.txt 2>&1")

t = threading.Thread(target=run)
t.daemon = True
t.start()

# Wait for it to start
time.sleep(15)

# Get URL
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print("Your app URL:", url)

In [ ]:
import os
import time
import threading

# Kill everything first
os.system("pkill -f streamlit")
os.system("pkill -f lt")
time.sleep(3)

# Start streamlit in background
def run():
    os.system("streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/logs.txt 2>&1")

t = threading.Thread(target=run)
t.daemon = True
t.start()

print("Waiting for Streamlit...")
time.sleep(15)

# Check if running
logs = os.popen("cat /content/logs.txt").read()
print(logs)

In [ ]:
import os
import time
import threading

# Install streamlit
os.system("pip install streamlit -q")
time.sleep(5)

# Kill existing
os.system("pkill -f streamlit")
time.sleep(2)

# Start streamlit in background
def run():
    os.system("streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/logs.txt 2>&1")

t = threading.Thread(target=run)
t.daemon = True
t.start()

print("Waiting for Streamlit to start...")
time.sleep(20)

# Check logs
print(os.popen("cat /content/logs.txt").read())

In [ ]:
app_code = """
import streamlit as st
import joblib
import pandas as pd

# Load model and scaler
model = joblib.load('diabetes_rf_model.pkl')
scaler = joblib.load('diabetes_scaler.pkl')

st.title('🏥 Diabetes Prediction App')
st.write('Enter patient details below to predict diabetes risk')

# Input fields
pregnancies = st.slider('Pregnancies', 0, 17, 3)
glucose = st.slider('Glucose Level', 44, 199, 117)
blood_pressure = st.slider('Blood Pressure', 24, 122, 72)
skin_thickness = st.slider('Skin Thickness', 7, 99, 23)
insulin = st.slider('Insulin', 14, 846, 30)
bmi = st.slider('BMI', 18.2, 67.1, 32.0)
dpf = st.slider('Diabetes Pedigree Function', 0.078, 2.42, 0.47)
age = st.slider('Age', 21, 81, 33)

if st.button('Predict'):
    input_data = pd.DataFrame([[pregnancies, glucose, blood_pressure,
                                skin_thickness, insulin, bmi, dpf, age]],
                               columns=['Pregnancies','Glucose','BloodPressure',
                                        'SkinThickness','Insulin','BMI',
                                        'DiabetesPedigreeFunction','Age'])
    input_scaled = scaler.transform(input_data)
    result = model.predict(input_scaled)[0]
    prob = model.predict_proba(input_scaled)[0]

    if result == 1:
        st.error(f'⚠️ Diabetes Detected! Confidence: {round(max(prob)*100, 2)}%')
    else:
        st.success(f'✅ No Diabetes! Confidence: {round(max(prob)*100, 2)}%')
"""

with open('app.py', 'w') as f:
    f.write(app_code)

print("app.py created! ✅")

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os, time, threading

def run():
    os.system("streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/logs.txt 2>&1")

t = threading.Thread(target=run)
t.daemon = True
t.start()

time.sleep(20)
print(os.popen("cat /content/logs.txt").read())

In [ ]:
!npm install -g localtunnel
!lt --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 5s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴your url is: https://quick-insects-jog.loca.lt


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import joblib
import pandas as pd

# Load model and scaler
model = joblib.load('diabetes_rf_model.pkl')
scaler = joblib.load('diabetes_scaler.pkl')

# Create sliders
pregnancies = widgets.IntSlider(min=0, max=17, value=3, description='Pregnancies:')
glucose = widgets.IntSlider(min=44, max=199, value=117, description='Glucose:')
blood_pressure = widgets.IntSlider(min=24, max=122, value=72, description='Blood Pressure:')
skin_thickness = widgets.IntSlider(min=7, max=99, value=23, description='Skin Thickness:')
insulin = widgets.IntSlider(min=14, max=846, value=30, description='Insulin:')
bmi = widgets.FloatSlider(min=18.2, max=67.1, value=32.0, description='BMI:')
dpf = widgets.FloatSlider(min=0.078, max=2.42, value=0.47, description='DPF:')
age = widgets.IntSlider(min=21, max=81, value=33, description='Age:')
button = widgets.Button(description='Predict', button_style='primary')
output = widgets.Output()

def on_predict(b):
    with output:
        output.clear_output()
        input_data = pd.DataFrame([[pregnancies.value, glucose.value, blood_pressure.value,
                                    skin_thickness.value, insulin.value, bmi.value,
                                    dpf.value, age.value]],
                                   columns=['Pregnancies','Glucose','BloodPressure',
                                            'SkinThickness','Insulin','BMI',
                                            'DiabetesPedigreeFunction','Age'])
        input_scaled = scaler.transform(input_data)
        result = model.predict(input_scaled)[0]
        prob = model.predict_proba(input_scaled)[0]

        if result == 1:
            print(f"⚠️ Diabetes Detected! Confidence: {round(max(prob)*100, 2)}%")
        else:
            print(f"✅ No Diabetes! Confidence: {round(max(prob)*100, 2)}%")

button.on_click(on_predict)

print("🏥 Diabetes Prediction App")
print("="*40)
display(pregnancies, glucose, blood_pressure, skin_thickness, insulin, bmi, dpf, age, button, output)